# Forest plot dos efeitos pareados do C-NBI

Este notebook resume os efeitos pareados do C-NBI sobre IGD e hipervolume nas frentes completas e com cardinalidade equalizada. A diferença é orientada para que valores positivos sempre favoreçam o C-NBI: para IGD, calcula-se o valor do concorrente menos o valor do C-NBI; para HV, calcula-se o valor do C-NBI menos o valor do concorrente. O ponto mostra a diferença mediana, a barra mostra o intervalo bootstrap percentil de 95% e o preenchimento indica significância após a correção de Holm.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
METRICS_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_metrics.csv'
STATISTICS_PATH = ROOT / 'results' / 'synthetic' / 'tables' / 'full_statistics.csv'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'paired_effects'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_WIDTH_CM = 16.0
CM_TO_INCH = 1 / 2.54
BOOTSTRAP_REPLICATES = 20_000
BOOTSTRAP_BASE_SEED = 20260827

COMPETITOR_ORDER = ['VRF-NBI', 'NSGA-III', 'MOEA/D', 'NBI original']
COMPETITOR_COLORS = {
    'VRF-NBI': '#0072B2',
    'NSGA-III': '#009E73',
    'MOEA/D': '#D62728',
    'NBI original': '#7B3294',
}
COMPETITOR_MARKERS = {
    'VRF-NBI': '^',
    'NSGA-III': 's',
    'MOEA/D': 'D',
    'NBI original': 'P',
}
SCENARIO_ORDER = [
    'm4_low', 'm4_medium', 'm4_high',
    'm6_low', 'm6_medium', 'm6_high',
    'm12_low', 'm12_medium', 'm12_high',
]
SCENARIO_LABELS = {
    'm4_low': r'$m=4$ — baixa',
    'm4_medium': r'$m=4$ — média',
    'm4_high': r'$m=4$ — alta',
    'm6_low': r'$m=6$ — baixa',
    'm6_medium': r'$m=6$ — média',
    'm6_high': r'$m=6$ — alta',
    'm12_low': r'$m=12$ — baixa',
    'm12_medium': r'$m=12$ — média',
    'm12_high': r'$m=12$ — alta',
}

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 8.5,
    'axes.titlesize': 9.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.2,
    'legend.fontsize': 7.2,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Métricas:', METRICS_PATH.relative_to(ROOT))
print('Estatísticas:', STATISTICS_PATH.relative_to(ROOT))

In [ ]:
metrics = pd.read_csv(METRICS_PATH)
statistics = pd.read_csv(STATISTICS_PATH)
statistics = statistics.loc[statistics['metric'].isin(['IGD', 'HV'])].copy()
statistics['competitor'] = statistics['method'].replace({'NBI': 'NBI original'})
statistics['version'] = statistics['comparison'].replace(
    {'complete': 'complete', 'equal_cardinality': 'equalized'}
)

def stable_seed(*parts):
    text = '|'.join(map(str, parts))
    return BOOTSTRAP_BASE_SEED + sum(
        (index + 1) * ord(character)
        for index, character in enumerate(text)
    )

def paired_favorable_differences(block, metric, competitor):
    wide = block.pivot(index='seed', columns='method', values=metric).sort_index()
    assert {'CNBI', competitor}.issubset(wide.columns)
    pair = wide[['CNBI', competitor]].dropna()
    assert len(pair) == 10
    if metric == 'IGD':
        differences = pair[competitor] - pair['CNBI']
    else:
        differences = pair['CNBI'] - pair[competitor]
    return differences.to_numpy(dtype=float)

rows = []
for scenario in SCENARIO_ORDER:
    for comparison, version in [('complete', 'complete'), ('equal_cardinality', 'equalized')]:
        block = metrics.loc[
            metrics['scenario'].eq(scenario)
            & metrics['comparison'].eq(comparison)
        ]
        available_methods = set(block['method'])
        competitors = ['VRF-NBI', 'NSGA-III', 'MOEA/D']
        if 'NBI' in available_methods:
            competitors.append('NBI')
        for metric in ['IGD', 'HV']:
            for competitor_raw in competitors:
                competitor = 'NBI original' if competitor_raw == 'NBI' else competitor_raw
                differences = paired_favorable_differences(block, metric, competitor_raw)
                rng = np.random.default_rng(
                    stable_seed(scenario, version, metric, competitor)
                )
                bootstrap_indices = rng.integers(
                    0, len(differences),
                    size=(BOOTSTRAP_REPLICATES, len(differences)),
                )
                bootstrap_medians = np.median(differences[bootstrap_indices], axis=1)
                ci_low, ci_high = np.percentile(bootstrap_medians, [2.5, 97.5])
                test = statistics.loc[
                    statistics['scenario'].eq(scenario)
                    & statistics['version'].eq(version)
                    & statistics['metric'].eq(metric)
                    & statistics['competitor'].eq(competitor)
                ]
                assert len(test) == 1
                test = test.iloc[0]
                rows.append({
                    'scenario': scenario,
                    'version': version,
                    'metric': metric,
                    'competitor': competitor,
                    'n_pairs': len(differences),
                    'median_favorable_difference': float(np.median(differences)),
                    'bootstrap_ci95_low': float(ci_low),
                    'bootstrap_ci95_high': float(ci_high),
                    'p_raw': float(test['p_raw']),
                    'p_holm': float(test['p_holm']),
                    'significant_holm_0p05': bool(test['p_holm'] < 0.05),
                })
effects = pd.DataFrame(rows)
assert len(effects) == 120
assert effects['n_pairs'].eq(10).all()
assert effects.loc[effects['competitor'].eq('NBI original'), 'scenario'].str.startswith('m4_').all()
effects_path = OUT_DIR / 'cnbi_paired_effects_bootstrap.csv'
effects.to_csv(effects_path, index=False)
effects.head()

In [ ]:
PANEL_SPECS = [
    ('IGD', 'complete', '(a) IGD\nfronteira completa'),
    ('IGD', 'equalized', '(b) IGD\ncardinalidade equalizada'),
    ('HV', 'complete', '(c) HV\nfronteira completa'),
    ('HV', 'equalized', '(d) HV\ncardinalidade equalizada'),
]
SCENARIO_POSITIONS = dict(zip(SCENARIO_ORDER, np.arange(len(SCENARIO_ORDER), dtype=float)))

def offsets_for_scenario(scenario):
    methods = COMPETITOR_ORDER if scenario.startswith('m4_') else COMPETITOR_ORDER[:-1]
    values = np.linspace(-0.24, 0.24, len(methods))
    return dict(zip(methods, values))

metric_limits = {}
for metric in ['IGD', 'HV']:
    subset = effects.loc[effects['metric'].eq(metric)]
    maximum = float(np.abs(subset[['bootstrap_ci95_low', 'bootstrap_ci95_high']].to_numpy()).max())
    metric_limits[metric] = (-1.08 * maximum, 1.08 * maximum)

figure, axes = plt.subplots(
    2, 2, sharey=True,
    figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 15.5 * CM_TO_INCH),
)
figure.subplots_adjust(
    left=0.255, right=0.99, top=0.95, bottom=0.17,
    hspace=0.34, wspace=0.22,
)

for axis, (metric, version, title) in zip(axes.flat, PANEL_SPECS):
    panel = effects.loc[
        effects['metric'].eq(metric) & effects['version'].eq(version)
    ]
    for scenario in SCENARIO_ORDER:
        center = SCENARIO_POSITIONS[scenario]
        offsets = offsets_for_scenario(scenario)
        for competitor, offset in offsets.items():
            row = panel.loc[
                panel['scenario'].eq(scenario)
                & panel['competitor'].eq(competitor)
            ]
            assert len(row) == 1
            row = row.iloc[0]
            estimate = float(row['median_favorable_difference'])
            lower_error = estimate - float(row['bootstrap_ci95_low'])
            upper_error = float(row['bootstrap_ci95_high']) - estimate
            significant = bool(row['significant_holm_0p05'])
            color = COMPETITOR_COLORS[competitor]
            axis.errorbar(
                estimate, center + offset,
                xerr=np.array([[lower_error], [upper_error]]),
                fmt=COMPETITOR_MARKERS[competitor], linestyle='none',
                color=color, markerfacecolor=color if significant else 'white',
                markeredgecolor=color, markeredgewidth=0.9,
                markersize=5.0, elinewidth=1.1, capsize=2.4, capthick=0.9,
                zorder=3,
            )

    axis.axvline(0, color='0.25', linewidth=0.8, linestyle='--', zorder=0)
    axis.set_title(title, pad=5)
    axis.set_xlim(*metric_limits[metric])
    axis.set_ylim(len(SCENARIO_ORDER) - 0.5, -0.5)
    axis.set_yticks(
        np.arange(len(SCENARIO_ORDER)),
        [SCENARIO_LABELS[scenario] for scenario in SCENARIO_ORDER],
    )
    axis.xaxis.set_major_locator(MaxNLocator(nbins=5))
    axis.grid(axis='x', alpha=0.18, linewidth=0.5)
    axis.set_axisbelow(True)
    for separator in (2.5, 5.5):
        axis.axhline(separator, color='0.55', linewidth=0.75)
    for spine in axis.spines.values():
        spine.set_color('0.35')
        spine.set_linewidth(0.65)

figure.supxlabel(
    '← concorrente favorecido     Diferença mediana favorável     C-NBI favorecido →',
    y=0.108, fontsize=8.5,
)
competitor_handles = [
    Line2D(
        [0], [0], linestyle='none', marker=COMPETITOR_MARKERS[competitor],
        markerfacecolor=COMPETITOR_COLORS[competitor],
        markeredgecolor=COMPETITOR_COLORS[competitor],
        markersize=5.2, label=competitor,
    )
    for competitor in COMPETITOR_ORDER
]
significance_handles = [
    Line2D([0], [0], linestyle='none', marker='o', markerfacecolor='0.25',
           markeredgecolor='0.25', markersize=5.2, label='significativo após Holm'),
    Line2D([0], [0], linestyle='none', marker='o', markerfacecolor='white',
           markeredgecolor='0.25', markersize=5.2, label='não significativo'),
]
figure.legend(
    handles=competitor_handles + significance_handles,
    loc='lower center', bbox_to_anchor=(0.5, 0.018),
    ncol=3, frameon=False, columnspacing=1.4, handletextpad=0.45,
)

figure_png = OUT_DIR / 'fig_forest_efeitos_pareados_cnbi_igd_hv.png'
figure_pdf = OUT_DIR / 'fig_forest_efeitos_pareados_cnbi_igd_hv.pdf'
figure.savefig(figure_png, dpi=300)
figure.savefig(figure_pdf, dpi=300)
plt.close(figure)
print('Figura:', figure_png.relative_to(ROOT))

In [ ]:
metadata = {
    'metrics_source': METRICS_PATH.relative_to(ROOT).as_posix(),
    'holm_source': STATISTICS_PATH.relative_to(ROOT).as_posix(),
    'metrics': ['IGD', 'HV'],
    'versions': ['complete', 'equalized'],
    'effect_orientation': {
        'IGD': 'competitor minus CNBI',
        'HV': 'CNBI minus competitor',
    },
    'positive_effect_favors': 'CNBI',
    'point': 'median paired favorable difference',
    'interval': 'paired percentile bootstrap 95% CI',
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'bootstrap_base_seed': BOOTSTRAP_BASE_SEED,
    'fill': 'p_holm < 0.05 from full_statistics.csv',
    'nbi_original_rule': 'included only in m=4 scenarios',
    'publication_width_cm': FIGURE_WIDTH_CM,
}
metadata_path = OUT_DIR / 'forest_efeitos_pareados_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
for artifact in [effects_path, figure_png, figure_pdf, metadata_path]:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Efeitos significativos após Holm:', int(effects['significant_holm_0p05'].sum()), 'de', len(effects))

## Leitura da figura

Valores à direita de zero favorecem o C-NBI; valores à esquerda favorecem o método concorrente. Marcadores preenchidos indicam significância após Holm ao nível de 5%, enquanto marcadores vazios indicam ausência de significância corrigida. O intervalo bootstrap descreve a incerteza da diferença mediana pareada e não substitui a decisão do teste de Wilcoxon corrigido por Holm.